# 00 · Setup and smoke test

Goal: check that the environment works, the raw files are complete and match the competition description,
and that `src/cv.py` (folds, metric, results logging) behaves as expected. No modelling, no plots.

In [ ]:
import sys, time, importlib
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import cv

cv.set_seed()
cv.ensure_dirs()
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

print("Python", sys.version.split()[0])
for m in ["pandas", "numpy", "scipy", "matplotlib", "seaborn", "sklearn", "statsmodels",
          "lightgbm", "xgboost", "catboost", "optuna", "shap", "pyarrow"]:
    print(f"{m:12s}", importlib.import_module(m).__version__)
print("ROOT:", ROOT)

In [ ]:
# All raw files present, with their sizes
for name, fname in cv.RAW_FILES.items():
    p = cv.DATA_RAW / fname
    print(f"{fname:25s}", f"{p.stat().st_size / 1e6:8.1f} MB" if p.exists() else "MISSING")
assert all((cv.DATA_RAW / f).exists() for f in cv.RAW_FILES.values()), "download the data first"

In [ ]:
# Load every file with the dtypes defined in cv.read_raw
t0 = time.perf_counter()
raw = {name: cv.read_raw(name) for name in cv.RAW_FILES}
print(f"loaded in {time.perf_counter() - t0:.1f}s\n")
for name, df in raw.items():
    print(f"{name:18s} {str(df.shape):16s} {df.memory_usage(deep=True).sum() / 1e6:7.1f} MB")
train, test = raw["train"], raw["test"]
train.head()

In [ ]:
print(train.dtypes, "\n")
print(test.dtypes)

In [ ]:
# Date ranges must match the competition description
def date_range(df):
    return df["date"].min().date(), df["date"].max().date(), df["date"].nunique()

print("train:", date_range(train))
print("test: ", date_range(test))
assert train["date"].min() == cv.TRAIN_START and train["date"].max() == cv.TRAIN_END
assert test["date"].min() == cv.TEST_START and test["date"].max() == cv.TEST_END
assert test["date"].nunique() == cv.HORIZON

In [ ]:
# Number of series and whether every date has every series
n_stores, n_fam = train["store_nbr"].nunique(), train["family"].nunique()
n_series = train.groupby(cv.KEYS, observed=True).ngroups
print(f"stores={n_stores}  families={n_fam}  series={n_series}  (stores x families = {n_stores * n_fam})")

rows_per_date = train.groupby("date").size()
print("rows per date:", rows_per_date.value_counts().to_dict())

all_days = pd.date_range(cv.TRAIN_START, cv.TRAIN_END, freq="D")
missing_days = all_days.difference(pd.DatetimeIndex(train["date"].unique()))
print(f"calendar days={len(all_days)}  days in train={train['date'].nunique()}  missing={len(missing_days)}")
print("missing days:", [d.date() for d in missing_days])

In [ ]:
# Test structure: every series once per test day, ids continue after train, matches sample_submission
sub = raw["sample_submission"]
print("test rows:", len(test), " expected:", n_series * cv.HORIZON)
print("test series == train series:",
      set(map(tuple, test[cv.KEYS].drop_duplicates().to_numpy())) == set(map(tuple, train[cv.KEYS].drop_duplicates().to_numpy())))
print("test ids:", test["id"].min(), "->", test["id"].max(), " train last id:", train["id"].max())
print("sample_submission ids == test ids:", sub["id"].equals(test["id"]))
assert len(test) == n_series * cv.HORIZON

In [ ]:
# Basic quality checks on train
print("missing values:\n", train.isna().sum().to_string(), "\n")
print("negative sales:", int((train["sales"] < 0).sum()))
print("share of zero sales:", round(float((train["sales"] == 0).mean()), 3))
print("sales max:", float(train["sales"].max()))
print("onpromotion range:", int(train["onpromotion"].min()), "->", int(train["onpromotion"].max()),
      "| test:", int(test["onpromotion"].min()), "->", int(test["onpromotion"].max()))

In [ ]:
# Side files: coverage of the train and test periods
oil, tr, hol, stores = raw["oil"], raw["transactions"], raw["holidays"], raw["stores"]

print("oil:          ", date_range(oil), "| missing prices:", int(oil["dcoilwtico"].isna().sum()))
print("transactions: ", date_range(tr), "| covers test period:", tr["date"].max() >= cv.TEST_START)
print("holidays:     ", date_range(hol), "| rows in test period:",
      int(hol["date"].between(cv.TEST_START, cv.TEST_END).sum()))
print("stores:       ", stores.shape, "| store ids match train:",
      set(stores["store_nbr"]) == set(train["store_nbr"]))
print("\nholiday types:", hol["type"].value_counts().to_dict())
print("holiday locales:", hol["locale"].value_counts().to_dict())
print("transferred:", hol["transferred"].value_counts().to_dict())

In [ ]:
# Backtest folds: defined once in src/cv.py
folds = cv.folds_table()
folds["valid_rows"] = [len(cv.split(train, f)[1]) for f in cv.FOLDS]
folds["train_rows"] = [len(cv.split(train, f)[0]) for f in cv.FOLDS]
print("test period:", cv.TEST_START.date(), cv.TEST_START.day_name(), "->", cv.TEST_END.date(), cv.TEST_END.day_name())
folds

In [ ]:
# Fold sanity: each window is 16 days, folds don't overlap, train always ends before its window
for f in cv.FOLDS:
    tr_f, va_f = cv.split(train, f)
    assert va_f["date"].nunique() == cv.HORIZON
    assert tr_f["date"].max() < va_f["date"].min()
assert all(a.valid_end < b.valid_start for a, b in zip(cv.FOLDS, cv.FOLDS[1:]))
assert cv.FOLDS[-1].valid_end == cv.TRAIN_END
print("folds OK")

In [ ]:
# Metric checks against hand-computed values
assert cv.rmsle([0, 1, 10], [0, 1, 10]) == 0.0
y, p = np.array([0.0, 3.0]), np.array([1.0, 1.0])
manual = np.sqrt(((np.log(2) - np.log(1)) ** 2 + (np.log(2) - np.log(4)) ** 2) / 2)
assert np.isclose(cv.rmsle(y, p), manual)
assert cv.rmsle([5.0], [-3.0]) == cv.rmsle([5.0], [0.0])   # negative predictions are clipped to 0
print("rmsle OK, example:", round(cv.rmsle(y, p), 5))

In [ ]:
# evaluate / log_result / save round trip, written to a temporary folder (results.csv is untouched)
import tempfile
tmp = Path(tempfile.mkdtemp())

# a dummy model: constant prediction of 1.0 on every validation row
dummy = pd.concat(
    [cv.split(train, f)[1][["date", *cv.KEYS]].assign(fold=f.fold, y_pred=1.0) for f in cv.FOLDS],
    ignore_index=True,
)
scores = cv.evaluate(dummy, train)
print("dummy fold scores:", {k: round(v, 4) for k, v in scores.items()})

cv.log_result("dummy_a", scores, params="c=1", notes="smoke test", path=tmp / "results.csv")
cv.log_result("dummy_b", {k: v + 0.1 for k, v in scores.items()}, path=tmp / "results.csv")
cv.log_result("dummy_a", scores, notes="re-run replaces the row", path=tmp / "results.csv")
res = cv.show_results(tmp / "results.csv")
assert len(res) == 2 and res.loc[0, "model"] == "dummy_a" and res.loc[0, "notes"] == "re-run replaces the row"
assert res.loc[1, "notes"] == ""                  # empty text stays empty, not NaN

(tmp / "empty.csv").touch()                       # an empty file behaves like a missing one
assert len(cv.log_result("x", scores, path=tmp / "empty.csv")) == 1

out = cv.save_backtest_preds("dummy_a", dummy, preds_dir=tmp)
back = cv.load_backtest_preds("dummy_a", preds_dir=tmp)
assert len(back) == len(dummy) and cv.evaluate(back, train) == scores

try:                                               # a model that drops rows must fail loudly
    cv.evaluate(dummy.iloc[:-5], train)
    raise AssertionError("evaluate should have failed")
except ValueError as e:
    print("missing rows caught:", e)
res

In [ ]:
print("results.csv exists:", cv.RESULTS_PATH.exists())
print("SMOKE TEST PASSED")

## Where I am

(To be written after the run.)